# Problem Setup
Here, we will fit a two-dimensional function using Unet and use convolution for the derivation of this two-dimensional function and compare it to the analytic form.

### U-net structure
![Unet](./Fig/Unet.png)

# Libraries

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from Lib.ImportLibrary import configure_notebook
from Lib import myFCNGrad as mg
from Lib.Unet import UNet5

dev = configure_notebook()
output_dir = Path("output/Case8")
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint = output_dir / "model.pt"

# Training parameters

In [ ]:
steps = 5000
lr = 1e-3

xmin = 0
ymin = 0

Length = 1
Height = 1

# ------------------------------ define domain and collocation points -------------------------------
resolution = 128

KernelSize = 5
pad_size = KernelSize//2

# Functions

In [ ]:
def f_exact(x, y):
    u0 = np.sin(np.pi * x) * np.sin(4 * np.pi * y) * np.cos(2 * np.pi * x) * np.cos(np.pi * y / 2)
    return u0
def df(x, y):
    du0dx = np.pi*np.sin(4*np.pi*y)*np.cos(np.pi*y/2)*(np.cos(np.pi*x)*np.cos(2*np.pi*x)-2*np.sin(np.pi*x)*np.sin(2*np.pi*x))
    du0dy = 1/2*np.pi*np.sin(np.pi*x)*np.cos(2*np.pi*x)*(8*np.cos(1/2*np.pi*y)*np.cos(4*np.pi*y)-np.sin(1/2*np.pi*y)*np.sin(4*np.pi*y))
    return du0dx, du0dy

# Generate data

In [ ]:
x = np.linspace(0,1,resolution)
y = np.linspace(0,1,resolution)
xGrid, yGrid = np.meshgrid(x, y)

f = f_exact(xGrid, yGrid)
dfx, dfy = df(xGrid, yGrid)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3,figsize=(15,5))
ax1 = axes[0].contourf(xGrid, yGrid, f, levels=100, cmap=plt.get_cmap('jet'))
axes[0].axis('equal')
axes[0].axis('off')
axes[0].set_title(r'$f(x)$')
cbar1=fig.colorbar(ax1, ax=axes[0])

ax2 = axes[1].contourf(xGrid, yGrid, dfx, levels=100, cmap=plt.get_cmap('jet'))
axes[1].axis('equal')
axes[1].axis('off')
axes[1].set_title(r'$\partial f(x)/\partial x$')
cbar1=fig.colorbar(ax2, ax=axes[1])

ax3 = axes[2].contourf(xGrid, yGrid, dfy, levels=100, cmap=plt.get_cmap('jet'))
axes[2].axis('equal')
axes[2].axis('off')
axes[2].set_title(r'$\partial f(x)/\partial y$')
cbar1=fig.colorbar(ax3, ax=axes[2])

# Defining the inputs to a function

Since Unet establishes an image-to-image link. Here we assume that the input image is f(x)/2.So here we establish the Unet relation from f->2f.

Here the dimension of the input image is [1,1,128,128] which is required for convolution where the dimension of the input is [N,C,W,H].
Where N represents the number of images, C represents the number of channels in the image, W,H represents the height and width (number of pixels) of the image respectively


In [ ]:
input_img = f/2
input_img = torch.from_numpy(input_img.reshape(1, 1, resolution, resolution)).float().to(dev)
print(input_img.shape)

output_img = torch.from_numpy(f.reshape(1, 1, resolution, resolution)).float().to(dev)

# Defining the Network

Where in_channels denotes the features of the input image, here we take a grayscale image as input, so the number of in_channels is 1, if the image is an RGB image, then the number of in_channels is 3; num_classes denotes the number of output images, here it is considered as one output.

In [ ]:
model = UNet5(in_channels=1, num_classes=1,channels = [32, 64, 64, 64, 64]).to(dev)

# Train Neural Network

In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=lr)

In [ ]:
start_time = time.time()
it_time = time.time()
for i in range(steps):
    optimizer.zero_grad()
    yh = model(input_img)
    loss = torch.mean(torch.square(yh-output_img)) # use mean squared error
    loss.backward()
    optimizer.step()
    if i % max(1, steps // 10) == 0:
      print('Iter: %d  Loss: %.5e  Time: %.3e'
                    % (i, loss.item(), time.time() - it_time))
      it_time = time.time()

torch.save(model.state_dict(), checkpoint)

# Test Model

In [ ]:
model.eval()
with torch.no_grad():
    output_y = model(input_img).cpu().numpy()
output_y = output_y.reshape(resolution, resolution)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2,figsize=(10,5))
ax1 = axes[0].contourf(xGrid, yGrid, f, levels=100, cmap=plt.get_cmap('jet'))
axes[0].axis('equal')
axes[0].axis('off')
axes[0].set_title(r'$f(x)$')
cbar1=fig.colorbar(ax1, ax=axes[0])

ax2 = axes[1].contourf(xGrid, yGrid, output_y, levels=100, cmap=plt.get_cmap('jet'))
axes[1].axis('equal')
axes[1].axis('off')
axes[1].set_title(r'$Pred f(x)$')
cbar1=fig.colorbar(ax1, ax=axes[1])

# Using convolution to obtain functional derivatives

In [ ]:
dx = x[1] - x[0]
dy = y[1] - y[0]
operators = mg.Derivative(dx=dx, dy=dy, KernelSize=KernelSize).to(dev)

Evaluate the analytical function at ghost points outside the grid to supply the stencil with boundary values. This checks the convolution against known derivatives of the reference field.

In [ ]:
offsets = np.arange(-pad_size, resolution + pad_size)
x_padded, y_padded = np.meshgrid(x[0] + offsets * dx, y[0] + offsets * dy)
padded_field = torch.as_tensor(f_exact(x_padded, y_padded), dtype=output_img.dtype, device=dev)[None, None]

In [ ]:
f_x = operators.dx(padded_field).reshape(resolution, resolution).detach().cpu().numpy()
f_y = operators.dy(padded_field).reshape(resolution, resolution).detach().cpu().numpy()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2,figsize=(10,5))
ax1 = axes[0].contourf(xGrid, yGrid, dfx, levels=100, cmap=plt.get_cmap('jet'))
axes[0].axis('equal')
axes[0].axis('off')
axes[0].set_title(r'$\partial f(x)/\partial x-True$')
cbar1=fig.colorbar(ax1, ax=axes[0])

ax2 = axes[1].contourf(xGrid, yGrid, f_x, levels=100, cmap=plt.get_cmap('jet'))
axes[1].axis('equal')
axes[1].axis('off')
axes[1].set_title(r'$\partial f(x)/\partial x$')
cbar1=fig.colorbar(ax1, ax=axes[1])


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2,figsize=(10,5))
ax1 = axes[0].contourf(xGrid, yGrid, dfy, levels=100, cmap=plt.get_cmap('jet'))
axes[0].axis('equal')
axes[0].axis('off')
axes[0].set_title(r'$\partial f(x)/\partial y-True$')
cbar1=fig.colorbar(ax1, ax=axes[0])

ax2 = axes[1].contourf(xGrid, yGrid, f_y, levels=100, cmap=plt.get_cmap('jet'))
axes[1].axis('equal')
axes[1].axis('off')
axes[1].set_title(r'$\partial f(x)/\partial y$')
cbar1=fig.colorbar(ax1, ax=axes[1])